# Question Answering for Private Documents

In [1]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), verbose=True)

True

In [2]:
!pip install -q pypdf


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip install -q docx2txt


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
!pip install -q wikipedia


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
def load_document(file):
    name, extension = os.path.splitext(file)
    if extension == ".pdf":
        from langchain_community.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == ".docx":
        from langchain_community.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    else:
        print(f'File extension {extension} not supported.')
        return None
    data = loader.load()
    return data

# wikipedia
def load_from_wikipedia(query, lang='en',load_max_docs=2):
    from langchain_community.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query = query, lang = lang, load_max_docs=load_max_docs)
    data = loader.load()
    return data


In [4]:
def chunk_data(data, chunk_size = 256, chunk_overlap = 50):
    from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    if isinstance(data, list):
        chunks = text_splitter.split_documents(data)
    else:
        chunks = text_splitter.split_text(data)
    return chunks


## Embedding

In [5]:
def insert_or_fetch_embeddings(index_name, chunks):
    from pinecone import Pinecone, ServerlessSpec
    from langchain_pinecone import PineconeVectorStore
    from langchain_community.embeddings import HuggingFaceEmbeddings

    # ✅ Initialize Pinecone with API key
    pc = Pinecone()

    # ✅ Load embedding model
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # ✅ Check if index exists
    if index_name in pc.list_indexes().names():
        print(f"Index '{index_name}' found. Loading...")

        vector_store = PineconeVectorStore.from_existing_index(
            index_name=index_name,
            embedding=embeddings
        )

    else:
        print(f"Index '{index_name}' not found. Creating new index...")

        pc.create_index(
            name=index_name,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

        # ✅ Insert documents ONLY when index is new
        vector_store = PineconeVectorStore.from_documents(
            documents=chunks,
            embedding=embeddings,
            index_name=index_name
        )

    return vector_store

In [6]:
def delete_pinecone_index(index_name = 'all'):
    from pinecone import Pinecone
    pc = Pinecone()
    if index_name == 'all':
        print(f"Deleting all indexes")
        for indexes in pc.list_indexes().names():
            pc.delete_index(indexes)
        print("Done")
    else:
        print(f"Deleting {index_name} index")
        pc.delete_index(index_name)
        print("Done")

In [7]:
def ask_and_get_answer(vector_store, q):
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains import RetrievalQA
    from langchain_groq import ChatGroq

    llm = ChatGroq(
        model = 'llama-3.3-70b-versatile',
        temperature= 1
    )

    retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':3})

    chain = RetrievalQA.from_chain_type(llm = llm, chain_type= "stuff",  retriever = retriever)

    answer = chain.invoke(q)
    return answer

In [8]:
data = load_document('files/uno.pdf')
print(f'You have {len(data)} pages in your document')

Loading files/uno.pdf
You have 9 pages in your document


In [10]:
data = load_document('files/dos.docx')
print(f'You have {len(data)} pages in your document')
print(data[0].page_content)

Loading files/dos.docx
You have 1 pages in your document
Demonstration of DOCX support in calibre

This document demonstrates the ability of the calibre DOCX Input plugin to convert the various typographic features in a Microsoft Word (2007 and newer) document. Convert this document to a modern ebook format, such as AZW3 for Kindles or EPUB for other ebook readers, to see it in action.

There is support for images, tables, lists, footnotes, endnotes, links, dropcaps and various types of text and paragraph level formatting.

To see the DOCX conversion in action, simply add this file to calibre using the “Add Books” button and then click “Convert”.  Set the output format in the top right corner of the conversion dialog to EPUB or AZW3 and click “OK”.



Text Formatting

Inline formatting

Here, we demonstrate various types of inline text formatting and the use of embedded fonts.

Here is some bold, italic, bold-italic, underlined and struck out  text. Then, we have a superscript and a su

In [11]:
data = load_from_wikipedia('GPT-4','de')
print(data[0].page_content)

OpenAI, Inc. ist ein US-amerikanisches Softwareunternehmen, das sich seit Ende 2015 mit der Erforschung künstlicher Intelligenz (KI, englisch Artificial Intelligence, AI) beschäftigt. Anfänglich war das Ziel von OpenAI, künstliche Intelligenz auf Open-Source-Basis zu entwickeln. Das Unternehmen war anfangs eine gemeinnützige Organisation ohne Gewinnerzielungsabsicht. 2019 wurde die gewinnorientierte Tochtergesellschaft OpenAI Global, LLC gegründet, in der Microsoft größter Investor ist. Die nicht gewinnorientierte Open AI Foundation hält einen Anteil von rund 26 Prozent an OpenAI Global LLC.
OpenAI ist vor allem bekannt für die Entwicklung der generativen vortrainierten Transformer (GPT) – auch generative künstliche Intelligenz, kurz GenAI, bezeichnet – und der daraus abgeleiteten Softwareprodukte wie ChatGPT oder DALL-E.


== Geschichte ==


=== Gründungsphase und Mission ===
Der Gründung von OpenAI im Jahr 2015 ging eine lange Debatte um die Risiken von KI voraus. Die Wissenschaftler

In [9]:
chunks = chunk_data(data)
print(len(chunks))
print(chunks[10].page_content)

73
Inessence,AIisthebroaderﬁeldencompassingthequesttocreateintelligentmachines,MLisasubsetfocusedonalgorithmsthatlearnfromdata,GenerativeAIspecializesincreatingnewdatasamples,NLPdealsspeciﬁcallywithhumanlanguageunderstandingandprocessing,andDLisasubsetofMLus


In [10]:
delete_pinecone_index()

Deleting all indexes
Done


In [11]:
index_name = 'askadocument'
vector_store = insert_or_fetch_embeddings(index_name, chunks)

/var/folders/ct/kw0llgf55jx6s41xt0dc6bb40000gn/T/ipykernel_65021/710583098.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index 'askadocument' not found. Creating new index...


In [15]:
q = "What is the whole document about?"
answer = ask_and_get_answer(vector_store, q)
print(answer)

{'query': 'What is the whole document about?', 'result': 'The document appears to be about Machine Learning (ML) applications, specifically categorizing and describing the different types of tasks and goals that ML can achieve.'}


In [16]:
import time
i = 1
print("Write quit or exit to quit")
while True:
    q = input("Enter your question: ")
    if q.lower() == 'exit' or q.lower() == 'quit':
        print("Quitting...")
        time.sleep(2)
        break
    answer = ask_and_get_answer(vector_store, q)
    print(f'\nAnswer: {answer}')
    print(f'\n{'-' * 50 }\n')

Write quit or exit to quit

Answer: {'query': 'what is the file about?', 'result': 'The file appears to be about Deep Learning (DL) and its relationship to Machine Learning (ML), specifically how deep learning models learn hierarchical representations of data.'}

--------------------------------------------------


Answer: {'query': 'so how is Deep Learning related to machine learning?', 'result': 'Deep Learning (DL) is a subset of Machine Learning (ML). This means that Deep Learning is a specific type of Machine Learning that utilizes artificial neural networks with multiple layers to model and solve complex problems. In other words, all Deep Learning is Machine Learning, but not all Machine Learning is Deep Learning.'}

--------------------------------------------------


Answer: {'query': 'what are neural networks?', 'result': 'Neural networks are a key component of Deep Learning (DL). They are artificial neural networks with multiple layers, also known as deep neural networks, that

In [17]:
delete_pinecone_index()

Deleting all indexes
Done


In [18]:
wiki = load_from_wikipedia('ChatGPT', 'ro')
chunks = chunk_data(wiki)
index_name = 'chatgpt'
vector_store = insert_or_fetch_embeddings(index_name, chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index 'chatgpt' not found. Creating new index...


In [19]:
q = 'Ce este ChatGPT?'
answer = ask_and_get_answer(vector_store, q)
print(answer)

{'query': 'Ce este ChatGPT?', 'result': 'ChatGPT este o versiune a modelului de inteligență artificială GPT-3, care este concepută pentru a genera texte și a imita conversații umane. El poate fi utilizat pentru o varietate de sarcini, cum ar fi:\n\n* Generarea de text pentru chatbot\n* Rezumarea datelor\n* Crearea de articole și relatari\n* Construirea sistemelor de asistență pentru clienți\n* Răspunsul la întrebări\n* Scrierea și depanarea programelor de calculator\n* Compunerea muzicii, scenariilor de televiziune, basmelor, eseurilor pentru studenți\n\nÎn general, ChatGPT este un instrument versatil care poate fi utilizat pentru a genera conținut și a imita conversații umane într-o varietate de contexte.'}


## Using ChromaDB as vector DB

In [20]:
!pip install -q chromadb


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [23]:
def create_embeddings_chroma(chunks, persist_directory = './chroma.db'):
    from langchain_classic.vectorstores import Chroma
    from langchain_huggingface import HuggingFaceEmbeddings

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=persist_directory
        )
    return vector_store


def load_embeddings_chroma(persist_directory = './chroma.db'):
    from langchain_classic.vectorstores import Chroma
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
    return vector_store



In [24]:
data = load_document('files/uno.pdf')
chunks = chunk_data(data)
vector_store = create_embeddings_chroma(chunks)

Loading files/uno.pdf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
q = "What is Machine Learning?"
answer = ask_and_get_answer(vector_store, q)
print(answer)

{'query': 'What is Machine Learning?', 'result': 'Machine Learning (ML) is a subset of Artificial Intelligence (AI) focused on developing algorithms that enable computers to learn from data and make predictions or decisions without being explicitly programmed.'}


## Adding Memory

In [28]:
from langchain_groq import ChatGroq
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain

llm = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature= 1
)
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':5})
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
chain = ConversationalRetrievalChain.from_llm(llm = llm, chain_type= "stuff",  retriever = retriever, memory = memory)

In [29]:
def ask_question(q,chain):
    result = chain.invoke({'question': q})
    return result

In [30]:
data = load_document('files/uno.pdf')
chunks = chunk_data(data)
vector_store = create_embeddings_chroma(chunks)

Loading files/uno.pdf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
q = "What is AI?"
answer = ask_question(q,chain)
print(answer['answer'])

According to the context, Artificial Intelligence (AI) involves the development of systems or machines capable of performing tasks that typically require human intelligence. It encompasses various techniques such as machine learning, natural language processing, computer vision, and robotics, and aims to simulate human-like intelligence to solve complex problems, make decisions, and interact with the environment.


In [37]:
q = "What all is parts are in it?"
answer = ask_question(q,chain)
print(answer['answer'])

The components or parts of Artificial Intelligence (AI) include:

1. Machine Learning (ML)
2. Natural Language Processing (NLP)
3. Computer Vision
4. Robotics

These techniquess are used to simulate human-like intelligence to solve complex problems, make decisions, and interact with the environment.


In [39]:
for item in answer['chat_history']:
    print(item)

content='What is AI?' additional_kwargs={} response_metadata={}
content='AI (Artificial Intelligence) involves the development of systems or machines capable of performing tasks that typically require human intelligence. It encompasses various techniques such as machine learning, natural language processing, computer vision, and robotics. The goal of AI is to simulate human-like intelligence to solve complex problems, make decisions, and interact with the environment.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='What is AI?' additional_kwargs={} response_metadata={}
content='Artificial Intelligence (AI) involves the development of systems or machines capable of performing tasks that typically require human intelligence. It encompasses various techniques such as machine learning, natural language processing, computer vision, and robotics. The goal of AI is to simulate human-like intelligence to solve complex problems, make decisions, and intera